# Parquet to parquet with a schema

We will do the following:
- download the CSV file from Hugging Face and convert to a parquet with a schema
- upload the parquet to Hugging Face


## Setup


In [ ]:
%%capture pkg_output
%%bash
apt-get update
apt-get install -y tree jq


In [ ]:
import requests
import pandas
import pandas as pd
import duckdb
from google.colab import userdata
import os


## Converting a remote CSV file to parquet



In [ ]:
url = 'https://huggingface.co/datasets/rwcitek/DUC-2026/resolve/main/sub-est2025.csv'
base_file = url.split("/")[-1].split(".")[0]
base_file += ".schema"
url, base_file


### Extract, transform, load ( ETL ) in one step


In [ ]:
# Connect to an in-memory database
con = duckdb.connect()

# Read the remote CSV and write it directly to a local Parquet file
con.execute(f"""
    COPY (
        SELECT *
        FROM read_csv(
          "{url}",
          encoding = 'latin-1',
          columns = {{
            'SUMLEV': 'VARCHAR',
            'STATE': 'VARCHAR',
            'COUNTY': 'VARCHAR',
            'PLACE': 'VARCHAR',
            'COUSUB': 'VARCHAR',
            'CONCIT': 'VARCHAR',
            'PRIMGEO_FLAG': 'UINT32',
            'FUNCSTAT': 'VARCHAR',
            'NAME': 'VARCHAR',
            'STNAME': 'VARCHAR',
            'ESTIMATESBASE2020': 'UINT32',
            'POPESTIMATE2020': 'UINT32',
            'POPESTIMATE2021': 'UINT32',
            'POPESTIMATE2022': 'UINT32',
            'POPESTIMATE2023': 'UINT32',
            'POPESTIMATE2024': 'UINT32',
            'POPESTIMATE2025': 'UINT32'
          }}
        )
    )
    TO '{base_file}.parquet'
    (FORMAT PARQUET);
""")

! ls -l {base_file}.parquet


## Verify


In [ ]:
duckdb.query(f'''
  select *
  from "{base_file}.parquet"
  limit 10
  ;
''').df()


In [ ]:
duckdb.query(f'''
  describe
  select *
  from "{base_file}.parquet"
  limit 10
  ;
''').df()


#### Milestone

We downloaded a CSV file from Hugging Face, converted it to parquet with a schema, and saved it on the Colab VM.

We now have a local copy of the parquet file that we can work with.


## Push to Hugging Face

### Configure settings and authenticate to Hugging Face


In [ ]:
hf_account = os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account


In [ ]:
hf_repo = os.environ["HF_REPO"] = "DUC-2026"
hf_repo


In [ ]:
hf_file = os.environ["HF_FILE"] = f"{base_file}.parquet"
hf_file


In [ ]:
# Retrieve the token from Colab's secret manager
_ = hf_token = os.environ["HF_TOKEN"] = userdata.get('hf_token')
f"{_[:3]} ... {_[-4:]}"


In [ ]:
# Log in automatically without an interactive prompt
! hf auth login --token ${HF_TOKEN}


## Push the parquet file to Hugging Face


In [ ]:
%%capture hf_upload
%%bash

TARGET_REPO="${HF_ACCOUNT}/${HF_REPO}"
LOCAL_SOURCE="./${HF_FILE}"
REMOTE_DESTINATION="./${HF_FILE}"

# Comment out the echo to make this live
# echo DRY-RUN: \
hf upload \
  --type dataset \
  "${TARGET_REPO}" \
  "${LOCAL_SOURCE}" \
  "${REMOTE_DESTINATION}"


In [ ]:
print(hf_upload.stdout)


## Query the parquet on Hugging Face using DuckDB


In [ ]:
# Construct the Hugging Face URL for your dataset file
hf_url = f"https://huggingface.co/datasets/{hf_account}/{hf_repo}/resolve/main/{hf_file}"
hf_url


In [ ]:
# Query the remote parquet
query = (f'''
  select count(1) as count
  from "{hf_url}"
  ;
''')
print(query)
duckdb.sql(query).df()


In [ ]:
# Query the remote parquet
query = (f'''
  describe
  select *
  from "{hf_url}"
  ;
''')
print(query)
duckdb.sql(query).df()


## Summary

We did the following:
- downloaded the CSV file from Hugging Face
- converted the CSV file to Parquet, with a schema
- queried the parquet file
- pushed the local parquet file to Hugging Face
- queried the remote parquet file on Hugging Face

